# Experiments – which recommender finds the best similar books?

Development notebook (not part of production). Production = `pipeline/pipeline.py` (item-item).

1. Train / validation / test split
2. Book features (genres, author, year, language, description vector)
3. Metrics
4. Models – popularity, item-item, ALS, two-tower
5. Training the two-tower model (validation after every epoch)
6. Test evaluation + Excel report

Inputs: `data/processed/` from the pipeline, `data/raw/` (tags, descriptions).
Outputs: `research/output/` – `results.xlsx`, `data_*.parquet` (split, book features), `model_*.pt` (trained models).

**Flags:** with `REBUILD_FEATURES = False` and `RETRAIN = False` existing outputs are loaded and the notebook runs in a few minutes.
Rebuilding features takes ~8 min, training one two-tower model ~45 min on CPU.

In [1]:
import os
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")   # implicit (ALS) is slow without this

import html
import json
import re
import sys
import time
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.sparse as sp
import torch
import torch.nn.functional as F
from torch import nn

sys.path.insert(0, str(Path("..") / "pipeline"))
from pipeline import item_similarity   # the same item-item as in production

RAW = Path("../data/raw")
DATA = Path("../data/processed")
OUT = Path("output")
OUT.mkdir(exist_ok=True)

REBUILD_FEATURES = False
RETRAIN = False
SEED = 42

## 1. Train / validation / test split

- Ratings **≥ 4** = "liked" – the model trains on these (user, book) pairs. The number itself is not used.
- Each user's liked books are split at random **80 / 10 / 10** (the ratings have no timestamp, so a time split is not possible).
  Users with fewer than 5 liked books stay in train only.
- **val** = tuning and choosing the best epoch, **test** = used once at the end.
- Ratings 1–3 (**disliked**) are not used for training, but the user has read those books, so they are never recommended in evaluation.

In [2]:
MIN_RATING, VAL_FRAC, TEST_FRAC, MIN_POSITIVES = 4, 0.1, 0.1, 5

interactions = pd.read_parquet(DATA / "interactions.parquet")
books = pd.read_parquet(DATA / "books.parquet").sort_values("book_id").reset_index(drop=True)
N_USERS, N_BOOKS = int(interactions.user_id.max()), int(interactions.book_id.max())


def to_index(df):
    """1-based ids -> 0-based indexes."""
    return pd.DataFrame({"user_idx": (df.user_id - 1).astype("int32"),
                         "book_idx": (df.book_id - 1).astype("int32")}).reset_index(drop=True)


def split_per_user(pos, seed=SEED):
    """Randomly split every user's liked books into train / val / test."""
    rng = np.random.default_rng(seed)
    df = pos.iloc[rng.permutation(len(pos))].reset_index(drop=True)
    rank = df.groupby("user_idx").cumcount().to_numpy()            # random position within the user
    n = df.groupby("user_idx")["user_idx"].transform("size").to_numpy()
    n_test = np.maximum(1, np.round(n * TEST_FRAC)).astype(int)
    n_val = np.maximum(1, np.round(n * VAL_FRAC)).astype(int)
    part = np.where(rank < n_test, "test", np.where(rank < n_test + n_val, "val", "train"))
    part[n < MIN_POSITIVES] = "train"
    return {name: df[part == name].sort_values(["user_idx", "book_idx"]).reset_index(drop=True)
            for name in ("train", "val", "test")}


parts = split_per_user(to_index(interactions[interactions.rating >= MIN_RATING]))
parts["disliked"] = to_index(interactions[interactions.rating < MIN_RATING])
pd.concat([df.assign(split=name) for name, df in parts.items()], ignore_index=True).to_parquet(OUT / "data_splits.parquet", index=False)

# checks: no (user, book) pair in two parts, nothing lost
pairs = {name: set(zip(df.user_idx, df.book_idx)) for name, df in parts.items()}
assert not pairs["train"] & pairs["val"] and not pairs["train"] & pairs["test"] and not pairs["val"] & pairs["test"]
assert sum(len(parts[n]) for n in ("train", "val", "test")) == (interactions.rating >= MIN_RATING).sum()

pd.DataFrame({"pairs": {n: len(d) for n, d in parts.items()}, "users": {n: d.user_idx.nunique() for n, d in parts.items()}})

,pairs,users
train,3297521,53406
val,412295,53366
test,412295,53366
disliked,1854368,52686


In [3]:
def to_csr(df):
    """User x book 0/1 matrix."""
    return sp.csr_matrix((np.ones(len(df), dtype=np.float32), (df.user_idx, df.book_idx)), shape=(N_USERS, N_BOOKS))


train, val, test, disliked = (to_csr(parts[n]) for n in ("train", "val", "test", "disliked"))
seen_val = train + disliked            # validation: never recommend what the user read
liked_test = train + val
seen_test = liked_test + disliked      # test: train + val + disliked are excluded

## 2. Book features

Used only by the two-tower model v2 (the book tower). Item-item and ALS cannot use them.

- **Genres from reader tags** (built in `pipeline/pipeline.py`, column `genres` in `books.parquet`) – statistical extract of the most frequent tags after removing shelves
  (`to-read`, `owned`, `kindle` …) → **30 genres**, each with its tag variants (`sci-fi`, `scifi`, `sf` → `science-fiction`).
  A book gets a genre when it has ≥ 30 % of the votes of its strongest genre (max 4).
- **Description vector** – descriptions (column `description`, from the Goodreads XML in goodbooks-10k) turned into 384 numbers by a local
  Hugging Face model `all-MiniLM-L6-v2`. The same model also guesses genres from the description, to compare with the tags.
- **Author** (first author; authors with 1 book share index 0), **year** (10 buckets), **language**.

In [4]:
from pipeline import GENRES   # genres (from reader tags) and descriptions are built in pipeline/pipeline.py

GENRE_LIST = list(GENRES)
YEAR_BINS = [1900, 1950, 1980, 1990, 2000, 2005, 2010, 2015]   # bucket 0 = unknown


def index_column(values, min_count=1):
    """Map values to 1..n by frequency; rare values and NaN get 0."""
    counts = values.value_counts()
    mapping = {v: i + 1 for i, v in enumerate(counts[counts >= min_count].index)}
    return values.map(mapping).fillna(0).astype(int)

In [5]:
FEATURES_FILE = OUT / "data_book_features.parquet"

if REBUILD_FEATURES or not FEATURES_FILE.exists():
    from sentence_transformers import SentenceTransformer

    feats = books[["book_id", "title", "author", "genres", "description", "original_publication_year", "language_code"]].copy()
    feats["has_description"] = feats.description.notna()
    texts = [f"{t}. {d if isinstance(d, str) else a}" for t, d, a in zip(feats.title, feats.description, feats.author)]
    encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device="cpu")
    desc_emb = encoder.encode(texts, batch_size=64, normalize_embeddings=True, show_progress_bar=True)
    genre_emb = encoder.encode([f"A {g.replace('-', ' ')} book." for g in GENRE_LIST], normalize_embeddings=True)
    feats["desc_genres"] = [[GENRE_LIST[i] for i in row] for row in np.argsort(-(desc_emb @ genre_emb.T), axis=1)[:, :2]]
    feats["desc_embedding"] = list(desc_emb.astype(np.float32))

    feats["author_main"] = feats.author
    feats["author_idx"] = index_column(feats.author_main, min_count=2)
    year = feats.original_publication_year.astype("float")
    feats["year_bucket"] = np.where(year.isna(), 0, np.digitize(year.fillna(0), YEAR_BINS) + 1)
    feats["language_idx"] = index_column(feats.language_code, min_count=10)
    feats.drop(columns=["author", "description", "original_publication_year", "language_code"]).to_parquet(FEATURES_FILE, index=False)

features = pd.read_parquet(FEATURES_FILE).sort_values("book_id").reset_index(drop=True)
agree = np.mean([d[0] in t for d, t in zip(features.desc_genres, features.genres)])
print(f"books with description: {features.has_description.sum()}, "
      f"description top genre agrees with tag genres: {agree:.0%}")
features.explode("genres").genres.value_counts().head(15)

books with description: 9685, description top genre agrees with tag genres: 33%


genres
fantasy               2032
romance               1958
young-adult           1744
non-fiction           1549
mystery               1437
thriller              1154
historical-fiction    1097
contemporary          1078
classics              1004
children               939
paranormal             773
science-fiction        772
biography-memoir       618
humor                  497
literary-fiction       432
Name: count, dtype: int64

In [6]:
features[["title", "genres", "desc_genres", "author_main", "year_bucket"]].head(10)

,title,genres,desc_genres,author_main,year_bucket
0,"The Hunger Games (The Hunger Games, #1)","[young-adult, dystopian, science-fiction]","[dystopian, politics]",Suzanne Collins,7
1,Harry Potter and the Sorcerer's Stone (Harry P...,"[fantasy, young-adult]","[fantasy, horror]",J.K. Rowling,5
2,"Twilight (Twilight, #1)","[young-adult, paranormal, fantasy]","[romance, thriller]",Stephenie Meyer,7
3,To Kill a Mockingbird,[classics],"[literary-fiction, children]",Harper Lee,3
4,The Great Gatsby,[classics],"[historical-fiction, classics]",F. Scott Fitzgerald,2
5,The Fault in Our Stars,"[young-adult, romance]","[romance, science-fiction]",John Green,8
6,The Hobbit,"[fantasy, classics]","[fantasy, classics]",J.R.R. Tolkien,2
7,The Catcher in the Rye,[classics],"[literary-fiction, young-adult]",J.D. Salinger,3
8,"Angels & Demons (Robert Langdon, #1)","[mystery, thriller]","[thriller, historical-fiction]",Dan Brown,6
9,Pride and Prejudice,[classics],"[romance, literary-fiction]",Jane Austen,1


## 3. Metrics

A recommender is a pair of scoring functions: `user_scores(users)` → scores of all books for each user, and
`book_scores(books)` → similarity of all books to each query book. Books the user already read are never recommended –
they get −∞ **before** the top K is taken, so the top K always has K unread books.

- **book_hit_rate@10** – **main metric**, the app scenario: one random liked book of the user is the query, 10 most similar
  books, hit if any of them is one of the user's hidden books.
- **recall@K** – share of the user's hidden books in their top K.
- **ndcg@K** – like recall, but a hit at rank 1 counts more than at rank K (weight 1 / log2(rank + 1)).
- **coverage@10** – share of all books recommended to at least one user.

In [7]:
def top_k(scores, exclude, k):
    """Indexes of the k best books per row, best first, skipping excluded books."""
    scores = np.array(scores, dtype=np.float32)
    rows, cols = exclude.nonzero()
    scores[rows, cols] = -np.inf
    idx = np.argpartition(-scores, k, axis=1)[:, :k]
    return np.take_along_axis(idx, np.argsort(-np.take_along_axis(scores, idx, axis=1), axis=1), axis=1)


def hits(top, targets):
    return np.take_along_axis(targets.toarray() > 0, top, axis=1)


def ndcg(hit, n_targets):
    k = hit.shape[1]
    discounts = 1.0 / np.log2(np.arange(2, k + 2))
    return (hit * discounts).sum(axis=1) / np.cumsum(discounts)[np.minimum(n_targets, k) - 1]


def rank_users(user_scores, exclude, users, k, batch=2048):
    return np.vstack([top_k(user_scores(users[s:s + batch]), exclude[users[s:s + batch]], k) for s in range(0, len(users), batch)])


def user_metrics(top, targets, users, ks=(10, 50)):
    """One row per user: hidden books, hits, recall@k, ndcg@k."""
    hit = np.vstack([hits(top[s:s + 2048], targets[users[s:s + 2048]]) for s in range(0, len(users), 2048)])
    n_targets = targets[users].getnnz(axis=1)
    df = pd.DataFrame({"user_idx": users, "n_targets": n_targets})
    for k in ks:
        df[f"hits@{k}"] = hit[:, :k].sum(axis=1)
        df[f"recall@{k}"] = hit[:, :k].sum(axis=1) / n_targets
        df[f"ndcg@{k}"] = ndcg(hit[:, :k], n_targets)
    return df


def book_to_book(book_scores, exclude, targets, liked, users, k=10, seed=0, batch=2048):
    """Per user: one random liked book is the query; hit if any of the k most similar books is a hidden book."""
    rng = np.random.default_rng(seed)
    users = users[liked[users].getnnz(axis=1) > 0]
    queries, hit = [], []
    for s in range(0, len(users), batch):
        b = users[s:s + batch]
        src = liked[b]
        q = np.array([rng.choice(src.indices[src.indptr[i]:src.indptr[i + 1]]) for i in range(len(b))])
        hit.append(hits(top_k(book_scores(q), exclude[b], k), targets[b]).any(axis=1))
        queries.append(q)
    return pd.DataFrame({"user_idx": users, "query_book_idx": np.concatenate(queries), "book_hit@10": np.concatenate(hit)})


def evaluate(model, exclude, targets, liked, ks=(10, 50)):
    """Average metrics + per-user details."""
    users = np.flatnonzero(targets.getnnz(axis=1))
    top = rank_users(model.user_scores, exclude, users, max(ks))
    per_user = user_metrics(top, targets, users, ks)
    b2b = book_to_book(model.book_scores, exclude, targets, liked, users)
    summary = {f"{m}@{k}": per_user[f"{m}@{k}"].mean() for k in ks for m in ("recall", "ndcg")}
    summary |= {"coverage@10": len(np.unique(top[:, :10])) / N_BOOKS, "book_hit_rate@10": b2b["book_hit@10"].mean()}
    return summary, (top, per_user, b2b)


# checks on tiny examples
assert top_k(np.array([[0.9, 0.8, 0.7, 0.1]]), sp.csr_matrix(([1.0], ([0], [0])), shape=(1, 4)), 2).tolist() == [[1, 2]]
assert np.isclose(ndcg(np.array([[True, False, True]]), np.array([2]))[0], (1 + 0.5) / (1 + 1 / np.log2(3)))

## 4. Models

| | How it finds similar books |
|---|---|
| Popularity | ignores the query – most liked books |
| Item-item | cosine of shared readers, no training (the production model, `pipeline.item_similarity`) |
| ALS | matrix factorization (`implicit`) – user and book vectors, cosine of book vectors |
| Two-tower | neural network – learns user and book vectors; v2 book tower also uses the book features |

In [8]:
class Popularity:
    def __init__(self, train):
        self.counts = np.asarray(train.sum(axis=0)).ravel().astype(np.float32)

    def user_scores(self, users):
        return np.tile(self.counts, (len(users), 1))

    book_scores = user_scores


class ItemItem:
    def __init__(self, train):
        self.train, self.sim = train, item_similarity(train)

    def user_scores(self, users):
        return self.train[users] @ self.sim

    def book_scores(self, books):
        return self.sim[books]


class ALS:
    """Chosen on val by book_hit_rate@10 (13 settings tried): 32 factors, regularization 0.01, alpha 40.
    128 factors / alpha 10 gives the best recall@10 (0.188) but only 0.21 book_hit_rate@10."""

    def __init__(self, train, factors=32, regularization=0.01, alpha=40.0, iterations=15):
        from implicit.cpu.als import AlternatingLeastSquares
        model = AlternatingLeastSquares(factors=factors, regularization=regularization, alpha=alpha,
                                        iterations=iterations, random_state=SEED)
        model.fit(train, show_progress=False)
        self.users, self.books = model.user_factors, model.item_factors
        self.unit = self.books / np.maximum(np.linalg.norm(self.books, axis=1, keepdims=True), 1e-9)

    def user_scores(self, users):
        return self.users[users] @ self.books.T

    def book_scores(self, books):
        return self.unit[books] @ self.unit.T

In [9]:
class Tower(nn.Module):
    """id -> embedding -> linear -> unit-length vector."""

    def __init__(self, n_ids, dim):
        super().__init__()
        self.embedding = nn.Embedding(n_ids, dim)
        self.proj = nn.Linear(dim, dim)
        nn.init.normal_(self.embedding.weight, std=0.1)

    def forward(self, ids):
        return F.normalize(self.proj(self.embedding(ids)), dim=-1)


class BookFeatureTower(nn.Module):
    """book id + author + year + language + genres + description -> MLP -> unit-length vector.
    Features are buffers indexed by book id, so the tower is still called with book ids only."""

    def __init__(self, n_books, dim, f):
        super().__init__()
        self.id_embedding = nn.Embedding(n_books, dim)
        self.author_embedding = nn.Embedding(int(f["author_idx"].max()) + 1, 32)
        self.year_embedding = nn.Embedding(len(YEAR_BINS) + 2, 8)
        self.language_embedding = nn.Embedding(int(f["language_idx"].max()) + 1, 8)
        self.genre_proj = nn.Linear(f["genres"].shape[1], 32, bias=False)
        self.desc_proj = nn.Linear(f["desc"].shape[1], 64)
        self.mlp = nn.Sequential(nn.Linear(dim + 32 + 8 + 8 + 32 + 64, 2 * dim), nn.ReLU(), nn.Linear(2 * dim, dim))
        nn.init.normal_(self.id_embedding.weight, std=0.1)
        for name in ("author_idx", "year_bucket", "language_idx", "genres", "desc"):
            self.register_buffer(name, f[name])

    def forward(self, ids):
        x = torch.cat([self.id_embedding(ids), self.author_embedding(self.author_idx[ids]),
                       self.year_embedding(self.year_bucket[ids]), self.language_embedding(self.language_idx[ids]),
                       self.genre_proj(self.genres[ids]), self.desc_proj(self.desc[ids])], dim=-1)
        return F.normalize(self.mlp(x), dim=-1)


class TwoTower(nn.Module):
    def __init__(self, dim=64, temperature=0.05, book_features=None):
        super().__init__()
        self.user_tower = Tower(N_USERS, dim)
        self.book_tower = Tower(N_BOOKS, dim) if book_features is None else BookFeatureTower(N_BOOKS, dim, book_features)
        self.temperature = temperature

    def forward(self, users, books):
        """Every user in the batch against every book in the batch (batch x batch)."""
        return self.user_tower(users) @ self.book_tower(books).T / self.temperature

    @torch.no_grad()
    def scorers(self):
        """Wrap the trained model in the user_scores / book_scores interface."""
        self.eval()
        book_emb = self.book_tower(torch.arange(N_BOOKS))
        user_scores = lambda u: (self.user_tower(torch.as_tensor(u)) @ book_emb.T).detach().numpy()
        book_scores = lambda b: (book_emb[torch.as_tensor(b)] @ book_emb.T).numpy()
        return type("Scorer", (), {"user_scores": staticmethod(user_scores), "book_scores": staticmethod(book_scores)})


def feature_tensors(f):
    multi_hot = np.zeros((len(f), len(GENRE_LIST)), dtype=np.float32)
    for row, genres in enumerate(f.genres):
        multi_hot[row, [GENRE_LIST.index(g) for g in genres]] = 1
    return {"author_idx": torch.as_tensor(f.author_idx.to_numpy()), "year_bucket": torch.as_tensor(f.year_bucket.to_numpy()),
            "language_idx": torch.as_tensor(f.language_idx.to_numpy()), "genres": torch.as_tensor(multi_hot),
            "desc": torch.as_tensor(np.stack(f.desc_embedding))}

## 5. Training the two-tower model

- **Loss: in-batch softmax.** For each (user, liked book) pair in a batch of 4 096, the other books in the batch are
  negatives; softmax over the row, loss = −log P(correct book). The same book twice in a batch is not a negative.
- **logQ correction:** popular books appear in batches more often → they would be pushed down as negatives too often.
  During training `score = cos(user, book) / τ − log P(book)`. Recommendations use plain cosine.
- Adam, lr 0.01 (1e-3 learned too slowly), up to 15 epochs, validation after every epoch, the best epoch is kept,
  early stopping after 2 epochs without improvement.
- v1 (ids only) was selected by recall@10, v2 (+ book features) by book_hit_rate@10.

In [10]:
def in_batch_loss(logits, books, log_q):
    logits = logits - log_q[books][None, :]                                  # logQ correction
    same = books[:, None] == books[None, :]
    same.fill_diagonal_(False)
    return F.cross_entropy(logits.masked_fill(same, float("-inf")), torch.arange(len(books)))


def train_two_tower(name, book_features=None, select="book_hit_rate@10", lr=0.01, epochs=15, batch=4096, patience=2):
    """Train (or load when it exists and RETRAIN is False).
    Saves output/model_<name>.pt with the best weights and the training history (val metrics per epoch)."""
    path = OUT / f"model_{name}.pt"
    model = TwoTower(book_features=book_features)
    if not RETRAIN and path.exists():
        checkpoint = torch.load(path)
        model.load_state_dict(checkpoint["state_dict"])
        return model, checkpoint["history"]

    torch.manual_seed(SEED)
    users = torch.as_tensor(parts["train"].user_idx.to_numpy(), dtype=torch.long)
    books_t = torch.as_tensor(parts["train"].book_idx.to_numpy(), dtype=torch.long)
    counts = torch.bincount(books_t, minlength=N_BOOKS).float()
    log_q = torch.log(counts.clamp(min=1) / counts.sum())
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    history, best, bad = [], -1.0, 0
    for epoch in range(1, epochs + 1):
        t0 = time.monotonic()
        model.train()
        order, total = torch.randperm(len(users)), 0.0
        for s in range(0, len(order), batch):
            i = order[s:s + batch]
            loss = in_batch_loss(model(users[i], books_t[i]), books_t[i], log_q)
            optimizer.zero_grad(); loss.backward(); optimizer.step()
            total += loss.item() * len(i)
        metrics, _ = evaluate(model.scorers(), seen_val, val, train, ks=(10,))
        history.append({"epoch": epoch, "loss": total / len(order), **metrics})
        print(f"epoch {epoch}: loss {total / len(order):.4f}, val recall@10 {metrics['recall@10']:.4f}, "
              f"book_hit_rate@10 {metrics['book_hit_rate@10']:.4f} ({time.monotonic() - t0:.0f}s)")
        if metrics[select] > best:
            best, bad = metrics[select], 0
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
        elif (bad := bad + 1) >= patience:
            break
    model.load_state_dict(best_state)
    torch.save({"state_dict": best_state, "history": history}, path)
    return model, history

In [11]:
two_tower_v1, history_v1 = train_two_tower("two_tower", select="recall@10")
two_tower_v2, history_v2 = train_two_tower("two_tower_features", book_features=feature_tensors(features))
pd.DataFrame(history_v2).round(4)

,epoch,loss,recall@10,ndcg@10,recall@50,ndcg@50,coverage@10,book_hit_rate@10
0,1,7.4780,0.1381,0.1434,0.3288,0.2225,0.6077,0.2666
1,2,6.8424,0.1632,0.1691,0.3761,0.2579,0.7528,0.2762
2,3,6.6506,0.1743,0.1798,0.3976,0.2734,0.8411,0.2800
3,4,6.5346,0.1810,0.1854,0.4100,0.2814,0.9068,0.2844
4,5,6.4523,0.1848,0.1890,0.4187,0.2871,0.9251,0.2857
5,6,6.3891,0.1866,0.1901,0.4230,0.2894,0.9399,0.2861
6,7,6.3383,0.1890,0.1921,0.4256,0.2915,0.9529,0.2883
7,8,6.2961,0.1906,0.1935,0.4307,0.2945,0.9589,0.2893
8,9,6.2602,0.1915,0.1942,0.4308,0.2948,0.9766,0.2908
9,10,6.2299,0.1912,0.1937,0.4320,0.2951,0.9694,0.2897


## 6. Test evaluation

Run once at the end. Excluded from recommendations: everything the user read (train + val + disliked).
Targets: the hidden test books.

In [12]:
models = {
    "Popularity": Popularity(train),
    "Item-item": ItemItem(train),
    "ALS": ALS(train),
    "Two-tower": two_tower_v1.scorers(),
    "Two-tower + features": two_tower_v2.scorers(),
}
results, details = {}, {}
for name, model in models.items():
    results[name], details[name] = evaluate(model, seen_test, test, liked_test)
summary = pd.DataFrame(results).T
summary.round(4)

C:\Users\Martin\uv_venv\uv_book_recommender\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,recall@10,ndcg@10,recall@50,ndcg@50,coverage@10,book_hit_rate@10
Popularity,0.0524,0.0598,0.1432,0.0977,0.0055,0.2891
Item-item,0.1754,0.2006,0.3624,0.2776,0.3781,0.3441
ALS,0.1373,0.1400,0.3543,0.2308,0.3121,0.2846
Two-tower,0.2098,0.2245,0.4350,0.3186,0.9776,0.2940
Two-tower + features,0.2122,0.2270,0.4419,0.3228,0.9812,0.2920


In [13]:
def similar_titles(model, title, n=5):
    q = books.index[books.title.str.startswith(title)][0]                 # books are sorted by popularity
    scores = model.book_scores(np.array([q]))[0]
    scores[q] = -np.inf
    return list(books.title.iloc[np.argsort(-scores)[:n]])


pd.DataFrame({name: similar_titles(m, "The Hobbit") for name, m in models.items() if name != "Popularity"})

,Item-item,ALS,Two-tower,Two-tower + features
0,The Fellowship of the Ring (The Lord of the Ri...,The Fellowship of the Ring (The Lord of the Ri...,The Fellowship of the Ring (The Lord of the Ri...,The Fellowship of the Ring (The Lord of the Ri...
1,Harry Potter and the Sorcerer's Stone (Harry P...,"The Lion, the Witch, and the Wardrobe (Chronic...","The Lion, the Witch, and the Wardrobe (Chronic...","The Lion, the Witch, and the Wardrobe (Chronic..."
2,"The Two Towers (The Lord of the Rings, #2)",Harry Potter and the Sorcerer's Stone (Harry P...,Animal Farm,1984
3,"The Return of the King (The Lord of the Rings,...",To Kill a Mockingbird,1984,Lord of the Flies
4,"The Lion, the Witch, and the Wardrobe (Chronic...",1984,To Kill a Mockingbird,Animal Farm


### Excel report – `output/results.xlsx`

| Sheet | Content |
|---|---|
| Souhrn | metrics per model + how much data was evaluated |
| Trenink | val metrics after every epoch |
| Uzivatele | metrics for every test user and model |
| Top50_vzorek | full top 50 for 100 random users, hits marked |
| Schovane_knihy | their hidden books and the rank each model gave them |
| Podobne_knihy | 10 most similar books for the 20 most popular books |

In [14]:
def write_report(path, n_sample=100, n_query=20):
    users = np.flatnonzero(test.getnnz(axis=1))
    setup = pd.DataFrame([
        ("vyhodnocených uživatelů (mají testové knihy)", len(users)),
        ("knih v katalogu", N_BOOKS),
        *[(f"{n} dvojic", len(parts[n])) for n in ("train", "val", "test", "disliked")],
        ("průměr schovaných knih na uživatele", round(len(parts["test"]) / len(users), 1)),
        ("z doporučení vyřazeno", "vše, co uživatel četl: train + val + hodnocení 1–3"),
    ], columns=["položka", "hodnota"])

    per_user = pd.DataFrame({"user_id": users + 1, "user_idx": users})
    for n in ("train", "val", "test", "disliked"):
        per_user[f"n_{n}"] = per_user.user_idx.map(parts[n].groupby("user_idx").size()).fillna(0).astype(int)
    for model, (_, m, b2b) in details.items():
        per_user = per_user.join(m.set_index("user_idx")[["recall@10", "recall@50", "hits@50", "ndcg@10"]].add_suffix(f" {model}"), on="user_idx")
        per_user[f"book_hit@10 {model}"] = per_user.user_idx.map(b2b.set_index("user_idx")["book_hit@10"]).astype(float)

    label = lambda idx: books.iloc[idx][["book_id", "title", "author"]].reset_index(drop=True)
    top50, hidden = [], []
    for pos in np.sort(np.random.default_rng(SEED).choice(len(users), n_sample, replace=False)):
        user, targets = users[pos], set(test[users[pos]].indices)
        rows = label(np.array(sorted(targets))).assign(user_id=user + 1)
        for model, (top, _, _) in details.items():
            t = label(top[pos]).assign(model=model, user_id=user + 1, rank=np.arange(1, top.shape[1] + 1))
            top50.append(t.assign(hit=np.where(np.isin(top[pos], list(targets)), "ano", "")))
            rank = {b: r + 1 for r, b in enumerate(top[pos])}
            rows[f"rank {model}"] = pd.array([rank.get(b) for b in sorted(targets)], dtype="Int64")
        hidden.append(rows)

    similar = [{"query": books.title.iloc[q], "rank": r + 1, **{name: similar_titles(m, books.title.iloc[q], 10)[r]
                for name, m in models.items() if name != "Popularity"}} for q in range(n_query) for r in range(10)]
    training = pd.concat([pd.DataFrame(history_v1).assign(model="Two-tower"),
                          pd.DataFrame(history_v2).assign(model="Two-tower + features")])

    with pd.ExcelWriter(path, engine="openpyxl") as xl:
        summary.round(4).to_excel(xl, sheet_name="Souhrn", index_label="model")
        setup.to_excel(xl, sheet_name="Souhrn", index=False, startrow=len(summary) + 3)
        training.round(4).to_excel(xl, sheet_name="Trenink", index=False)
        per_user.round(4).to_excel(xl, sheet_name="Uzivatele", index=False)
        pd.concat(top50)[["model", "user_id", "rank", "book_id", "title", "author", "hit"]].to_excel(xl, sheet_name="Top50_vzorek", index=False)
        pd.concat(hidden).to_excel(xl, sheet_name="Schovane_knihy", index=False)
        pd.DataFrame(similar).to_excel(xl, sheet_name="Podobne_knihy", index=False)
        for sheet in xl.sheets.values():
            for col in sheet.columns:
                width = max(len(str(c.value)) if c.value is not None else 0 for c in col[:200])
                sheet.column_dimensions[col[0].column_letter].width = min(max(width + 2, 8), 60)
            sheet.freeze_panes = "A2"


try:
    write_report(OUT / "results.xlsx")
except PermissionError:
    write_report(OUT / f"results_{time.strftime('%Y%m%d_%H%M%S')}.xlsx")   # results.xlsx is open in Excel
print("report saved")

report saved
